# Clase 3 · RAG a mano (ejercicio de casa)

> **Audiencia:** técnicos del núcleo (~4) y Champions con base de software que quieran
> construir el pipeline de RAG en Python plano — el espejo de `agente-transparente/loop.py`
> de la clase 2, ahora con RAG.
>
> **No es obligatorio para la asistencia a S4-S7.** No se evalúa. Es de casa.
>
> **Ver spec 06.** Stack: `langchain` + `langchain-openai` + `chromadb`. Credenciales
> Azure OpenAI ya usadas en la clase 2.


## Sección 0 · Preparación

In [1]:
# Verificación de entorno: Python 3.12, langchain, chroma, .env, conectividad.
import sys, os
from pathlib import Path
assert sys.version_info >= (3, 10), f"Se requiere Python >= 3.10, tenes {sys.version_info.split()[0]}"

try:
    import langchain
    import langchain_openai
    import chromadb
except ImportError as e:
    print(f"Falta dependencia: {e}")
    print("  Instala con: pip install -r notebook/requirements_notebook.txt")
    raise

# .env
from dotenv import load_dotenv
load_dotenv()
required = ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY",
            "AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "AZURE_OPENAI_DEPLOYMENT"]
faltan = [v for v in required if not os.getenv(v)]
assert not faltan, f"Faltan vars de entorno: {falten}"
print(".env cargado -", os.getenv("AZURE_OPENAI_DEPLOYMENT"))

# Conectividad - Bios usa Azure AI Foundry (no Azure OpenAI clasico).
# El endpoint *.services.ai.azure.com/openai/v1 se consume con OpenAI(base_url=...)
# igual que la clase 2 (ver agente-transparente/cliente.py).
from openai import OpenAI
try:
    cliente = OpenAI(
        base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    )
    r = cliente.embeddings.create(
        model=os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
        input="hola"
    )
    print(f"Conectividad OK: dim {len(r.data[0].embedding)}")
except Exception as e:
    print(f"No responde: {e}")
    raise

print("\nTodo listo. Empeza por la seccion 1.")


.env cargado - gpt-4o-mini
Conectividad OK: dim 1536

Todo listo. Empeza por la seccion 1.


In [5]:
pip install -r notebook/requirements_notebook.txt



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'notebook/requirements_notebook.txt'
Note: you may need to restart the kernel to use updated packages.


## Sección 1 · Cargar el corpus Bios

Vamos a usar el corpus sintético de la clase 3 (el mismo de la app visual).
No lo cargamos desde `app/data/chunks.json` porque ese ya viene chunked —
queremos hacer el chunking acá, paso a paso.


In [3]:
# Cargar corpus.json (lo produce scripts/generar_corpus.py)
import json
from pathlib import Path

# Buscar el corpus desde varias posibles ubicaciones del CWD
CORPUS_PATH = Path("../app/data/corpus.json")
if not CORPUS_PATH.exists():
    CORPUS_PATH = Path("app/data/corpus.json")
if not CORPUS_PATH.exists():
    import os
    for base in [Path(os.getcwd()), Path(os.getcwd()).parent]:
        cand = base / "app" / "data" / "corpus.json"
        if cand.exists():
            CORPUS_PATH = cand
            break
assert CORPUS_PATH.exists(), f"No se encontro corpus.json"

corpus = json.loads(CORPUS_PATH.read_text(encoding="utf-8"))
print(f"Familias: {len(corpus['familias'])}")
print(f"Documentos: {corpus['_stats']['documentos']}")
print(f"Parrafos: {corpus['_stats']['paragrafos']}")
print(f"Palabras: {corpus['_stats']['palabras_aprox']}")
print(f"Anomalias plantadas: {len(corpus['anomalias_plantadas'])}")
for an in corpus['anomalias_plantadas']:
    print(f"    - {an['id']}: {an['doc']} (parrafo {an['parrafo_idx']})")


Familias: 4
Documentos: 12
Parrafos: 37
Palabras: 1649
Anomalias plantadas: 3
    - RAG-1: mant_guia_sensor (parrafo 1)
    - RAG-2: comp_politica_abastecimiento (parrafo 2)
    - RAG-3: logi_manual_despachos (parrafo 2)


## Sección 2 · Chunking

`RecursiveCharacterTextSplitter` de langchain: parte en chunks de ~500 tokens
con 50 de solapamiento.

### `# TODO` (2 líneas)

Tenés que declarar el splitter con los parámetros correctos. La docstring
del splitter dice que `chunk_size` está en caracteres (no tokens), y que
`chunk_overlap` debe ser menor que `chunk_size`.

Para ~500 tokens en español, ~2400 caracteres es un buen `chunk_size`.


In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# TODO (1/2): instanciá el splitter con chunk_size=2400 y chunk_overlap=240
splitter = RecursiveCharacterTextSplitter(chunk_size=2400, chunk_overlap=240)

# TODO (2/2): aplicá el splitter a cada documento del corpus
# (cada documento es el join de sus párrafos con \n\n)
docs_texto = []
docs_meta = []
for fam in corpus['familias']:
    for doc in fam['documentos']:
        texto = "\n\n".join(doc['parrafos'])
        for chunk in splitter.split_text(texto):
            docs_texto.append(chunk)
            docs_meta.append({
                "doc_fuente": doc['id'],
                "dominio": fam['id'],
                "dominio_label": fam['dominio']
            })

print(f"✓ {len(docs_texto)} chunks generados")

# Verificación del sistema
assert len(docs_texto) >= len(corpus['familias']), "Algo raro: menos chunks que documentos"
assert all(len(t) > 50 for t in docs_texto), "Hay chunks muy cortos (¿chunk_size mal?)"
print(f"  · chunk más corto: {min(len(t) for t in docs_texto)} chars")
print(f"  · chunk más largo: {max(len(t) for t in docs_texto)} chars")


✓ 12 chunks generados
  · chunk más corto: 702 chars
  · chunk más largo: 1019 chars


## Sección 3 · Embeddings

`AzureOpenAIEmbeddings` de langchain-openai. Mismo modelo que el pre-bakeo.

> **Cuidado con el costo.** Embedar los 200 chunks de una sola vez es
> ~0.01 USD con text-embedding-3-small. No es catastrófico, pero no lo
> corras 100 veces para probar cosas — el cacheo es responsabilidad tuya.


In [5]:
from langchain_openai import OpenAIEmbeddings

# Bios usa Azure AI Foundry: el endpoint se pasa como base_url.
# El deployment name se pasa como model. Igual que clase 2.
embeddings = OpenAIEmbeddings(
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    model=os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
)

# Embeda los primeros 5 chunks como prueba
vectors_demo = embeddings.embed_documents(docs_texto[:5])
print(f"5 chunks embedados - dim {len(vectors_demo[0])}")
print(f"  primeros 5 valores: {vectors_demo[0][:5]}")


5 chunks embedados - dim 1536
  primeros 5 valores: [0.0033969879150390625, 0.0767822265625, 0.011260986328125, -0.0182342529296875, -0.03497314453125]


## Sección 4 · Chroma index

`Chroma.from_documents`: indexa en local, persistente entre runs.

### `# TODO` (3 líneas)

Tenés que pasar los chunks como `langchain_core.documents.Document` y dejar
que Chroma persista en `./chroma_clase3/`.


In [6]:
from langchain_core.documents import Document
from langchain_chroma import Chroma

# TODO (1/3): construís la lista de Document a partir de docs_texto + docs_meta
documents = [
    Document(page_content=texto, metadata=meta)
    for texto, meta in zip(docs_texto, docs_meta)
]

# TODO (2/3): instanciás Chroma con persist_directory="./chroma_clase3"
# (usar from_documents con embedding_function=embeddings)
import chromadb
client = chromadb.PersistentClient(path="./chroma_clase3")

# TODO (3/3): creás el vector store
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./chroma_clase3",
    collection_name="bios_clase3"
)

print(f"✓ Chroma indexado: {vectorstore._collection.count()} chunks")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✓ Chroma indexado: 12 chunks


## Sección 5 · Retrieval con coseno

Chroma hace el coseno internamente; `similarity_search_with_score` te da
los chunks más relevantes + su score.

### `# TODO` (3 líneas)

Probá con Q2 (Compras) — la pregunta semilla de la spec 03.


In [7]:
# TODO (1/3): definir la query Q2
query_q2 = corpus['preguntas_semilla'][1]['texto']
print(f"Q2: {query_q2}\n")

# TODO (2/3): llamá similarity_search_with_score con k=5
resultados = vectorstore.similarity_search_with_score(query_q2, k=5)

# TODO (3/3): iterá y muestra los resultados
for i, (doc, score) in enumerate(resultados):
    print(f"  {i+1}. score={score:.4f} · {doc.metadata['doc_fuente']}")
    print(f"     {doc.page_content[:120]}...\n")

# Validación: el chunk con anomalia RAG-2 (comp_politica_abastecimiento)
# debería estar en el top-3.
docs_top3 = [doc.metadata['doc_fuente'] for doc, _ in resultados[:3]]
assert 'comp_politica_abastecimiento' in docs_top3, \
    f"El chunk plantado no apareció en top-3: {docs_top3}"
print(f"✓ Anomalía RAG-2 encontrada en top-3")


Q2: Si la planta de Itagüí declara stock crítico de maíz amarillo, ¿qué procedimiento sigue el área de Compras para reabastecer?



Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  1. score=0.6824 · comp_reglas_inventario_minimo
     El inventario mínimo de una materia prima en una planta es la cantidad por debajo de la cual se considera stock crítico....

  2. score=0.6864 · comp_politica_abastecimiento
     El abastecimiento de materias primas a las plantas se rige por dos principios: cobertura mínima por planta y priorizació...

  3. score=0.8368 · prod_procedimiento_planeacion_demanda
     La planeación de demanda se actualiza mensualmente. El insumo principal es la proyección estadística generada por el áre...

  4. score=0.8941 · comp_manual_volumenes
     La planeación de volúmenes de materia prima por planta se actualiza mensualmente. El insumo principal es la proyección d...

  5. score=0.9727 · mant_manual_preventivo
     El mantenimiento preventivo de la planta es responsabilidad de la Coordinación de Mantenimiento. Toda intervención progr...

✓ Anomalía RAG-2 encontrada en top-3


## Sección 6 · Generación

El LLM recibe el top-K como contexto y la pregunta. Responde con cita.

### `# TODO` (4 líneas)

Vas a:
1. Construir un prompt template con `context` y `question`.
2. Pasárselo al LLM (`AzureChatOpenAI`).
3. Ejecutar.
4. Mostrar el output.


In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# TODO (1/4): instanciar el LLM
# Bios usa Azure AI Foundry: ChatOpenAI con base_url (igual que clase 2).
llm = ChatOpenAI(
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
    temperature=0.2,
)

# TODO (2/4): construir el prompt template
template = '''Eres un agente de operaciones de Grupo Bios. Respondes en espanol.
Cuando uses el contexto recuperado, citas el documento fuente.
Si el contexto no responde la pregunta, dices "no encontre documentacion sobre eso".

Contexto recuperado:
{context}

Pregunta: {question}

Respuesta (con cita):'''
prompt = ChatPromptTemplate.from_template(template)

# TODO (3/4): construir la cadena y ejecutar
context = "\n\n".join(
    f"[{doc.metadata['doc_fuente']}] {doc.page_content}"
    for doc, _ in resultados[:4]
)
chain = prompt | llm | StrOutputParser()

# TODO (4/4): mostrar el output
respuesta = chain.invoke({"context": context, "question": query_q2})
print("Respuesta del agente:\n")
print(respuesta)


NotFoundError: Error code: 404 - {'error': {'type': 'invalid_request_error', 'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}}

## Sección 7 · Citas — verificación

Para que el output sea realmente RAG (no solo LLM), la respuesta debe traer
la cita al documento fuente. Vamos a extraerlas y mostrarlas como JSON,
espejo del contrato del webhook de n8n.

### `# TODO` (2 líneas)


In [ ]:
# TODO (1/2): construir un dict con la respuesta y las citas
salida = {
    "respuesta": respuesta,
    "citas": [
        {
            "doc": doc.metadata['doc_fuente'],
            "chunk_texto": doc.page_content[:80] + "...",
            "score": float(score)
        }
        for doc, score in resultados[:4]
    ]
}

# TODO (2/2): imprimir como JSON pretty
print(json.dumps(salida, indent=2, ensure_ascii=False))


## Sección 8 · Cierre y puente

Lo que acabás de construir es lo que hace el workflow de n8n de la clase 3,
en Python plano. Si querés:

- **Armar un agente con memoria**: agregá las piezas de la clase 2
  (`memoria.py`, `loop.py`, `agente.py`). Lo que viste hoy + lo de S2 = un
  agente que lee documentos y tiene conversación multi-turno.
- **Exponerlo por webhook**: envolve la lógica de `respuesta = chain.invoke(...)`
  en un endpoint FastAPI con `POST /ask` (no se hace aquí — eso ya es
  invento propio del proyecto del Champion).
- **Avanzar**: reranking, hybrid search, query rewriting son extensiones.
  Las trabajamos en el acompañamiento S5-S7, no como bloque docente.

> **Spec 06:** este notebook es ejercicio de casa, no de clase. Si lo
> corriste entero, ya sabés construir RAG en código plano. Eso es el
> espejo de lo que vieron en la app visual y en el workflow de n8n.
